In [1]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn networkx neo4j==6.4.1 tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.7/390.7 kB 9.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn
import networkx as nx
import neo4j

print("Semua library utama berhasil diimpor!")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("networkx:", nx.__version__)
print("neo4j:", neo4j.__version__)

Semua library utama berhasil diimpor!
pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.6.1
networkx: 3.7
neo4j: 6.4.1


PEMERIKSAAN DATASET

In [3]:
import pandas as pd
from pathlib import Path

# Prioritaskan dataset asli pada laptop Rima; /content hanya fallback untuk Colab.
candidates = [Path(r'C:/Users/RIMA/UTS-GSD/data/events.csv'), Path('data/events.csv'), Path('events.csv'), Path('/content/events.csv')]
csv_path = next((p for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError('events.csv tidak ditemukan. Upload ke /content/events.csv atau taruh di data/events.csv')

nama_kolom = ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
# Kenali CSV ber-header dan tanpa header tanpa menghilangkan baris pertama.
with csv_path.open('r', encoding='utf-8-sig') as f:
    first = f.readline().strip().split(',')
header_ada = [x.strip().lower() for x in first] == nama_kolom

events = pd.read_csv(csv_path, header=0 if header_ada else None,
                     names=nama_kolom, low_memory=False)
for kolom in ['timestamp', 'visitorid', 'itemid', 'transactionid']:
    events[kolom] = pd.to_numeric(events[kolom], errors='coerce')
events['event'] = events['event'].astype('string').str.strip().str.lower()

print('Baris sebelum pembersihan:', len(events))
print('Nilai kosong sebelum pembersihan:')
print(events.isna().sum())
# Hanya kolom wajib yang harus terisi; transactionid memang boleh kosong.
events = events.dropna(subset=['timestamp', 'visitorid', 'event', 'itemid']).copy()
events = events[events['event'].isin(['view', 'addtocart', 'transaction'])].copy()
for kolom in ['timestamp', 'visitorid', 'itemid']:
    events[kolom] = events[kolom].astype('int64')
events.reset_index(drop=True, inplace=True)

print('Ukuran dataset:', events.shape)
print('Nama kolom:', events.columns.tolist())
display(events.head())
events.info()
print('Distribusi event:')
print(events['event'].value_counts())


Baris sebelum pembersihan: 552019
Nilai kosong sebelum pembersihan:
timestamp             0
visitorid             1
event                 1
itemid                1
transactionid    547424
dtype: int64
Ukuran dataset: (552018, 5)
Nama kolom: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552018 entries, 0 to 552017
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      552018 non-null  int64  
 1   visitorid      552018 non-null  int64  
 2   event          552018 non-null  string 
 3   itemid         552018 non-null  int64  
 4   transactionid  4595 non-null    float64
dtypes: float64(1), int64(3), string(1)
memory usage: 21.1 MB
Distribusi event:
event
view           533645
addtocart       13778
transaction      4595
Name: count, dtype: Int64


PERIKSA RENTANG WAKTU DATASET

In [4]:

import pandas as pd

# Periksa rentang timestamp asli.
print("Timestamp terkecil:", events["timestamp"].min())
print("Timestamp terbesar:", events["timestamp"].max())

# RetailRocket umumnya menggunakan Unix epoch dalam milidetik.
events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    errors="coerce",
    utc=True
)

print("\nTanggal paling awal:", events["datetime"].min())
print("Tanggal paling akhir:", events["datetime"].max())

print("\nJumlah timestamp yang gagal dikonversi:",
      events["datetime"].isna().sum())


Timestamp terkecil: 1433138409198
Timestamp terbesar: 1435557556827

Tanggal paling awal: 2015-06-01 06:00:09.198000+00:00
Tanggal paling akhir: 2015-06-29 05:59:16.827000+00:00

Jumlah timestamp yang gagal dikonversi: 0


PERIKSA DISTRIBUSI DATA PER BULAN

In [5]:
events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    utc=True
)

events["month"] = events["datetime"].dt.strftime("%Y-%m")

monthly_counts = (
    events.groupby("month")
    .size()
    .rename("jumlah_event")
    .to_frame()
)

display(monthly_counts)

,jumlah_event
month,
2015-06,552018


FILTER DAN SIMPAN CSV UNTUK P1 DAN P2 (TRAINING 23/05/2015–11/08/2015; TESTING 12/08/2015–01/09/2015)

In [6]:
import pandas as pd

# Semua batas waktu menggunakan UTC; tanggal akhir termasuk sampai 23:59:59.999.
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms", errors="coerce", utc=True)

# P1: TRAINING 23/05/2015 s.d. 11/08/2015 (inklusif)
p1_mulai = pd.Timestamp("2015-05-23", tz="UTC")
p1_akhir_eksklusif = pd.Timestamp("2015-08-12", tz="UTC")
p1 = events.loc[(events["datetime"] >= p1_mulai) &
                (events["datetime"] < p1_akhir_eksklusif)].copy()

# P2: TESTING 12/08/2015 s.d. 01/09/2015 (inklusif)
p2_mulai = pd.Timestamp("2015-08-12", tz="UTC")
p2_akhir_eksklusif = pd.Timestamp("2015-09-02", tz="UTC")
p2 = events.loc[(events["datetime"] >= p2_mulai) &
                (events["datetime"] < p2_akhir_eksklusif)].copy()

assert p1.empty or p1["datetime"].max() < p2_mulai
assert p2.empty or p2["datetime"].min() >= p2_mulai
assert len(p1) + len(p2) <= len(events)

p1.drop(columns=["datetime"], errors="ignore").to_csv(str(csv_path.parent / "events_P1_training.csv"), index=False)
p2.drop(columns=["datetime"], errors="ignore").to_csv(str(csv_path.parent / "events_P2_testing.csv"), index=False)

for nama, df in [("P1 TRAINING (23/05–11/08/2015)", p1),
                 ("P2 TESTING (12/08–01/09/2015)", p2)]:
    print(f"\n=== {nama} ===")
    print("Jumlah event:", len(df))
    print("Distribusi event:\n", df["event"].value_counts())
    print("Tanggal aktual:", df["datetime"].min(), "s.d.", df["datetime"].max())



=== P1 TRAINING (23/05–11/08/2015) ===
Jumlah event: 552018
Distribusi event:
 event
view           533645
addtocart       13778
transaction      4595
Name: count, dtype: Int64
Tanggal aktual: 2015-06-01 06:00:09.198000+00:00 s.d. 2015-06-29 05:59:16.827000+00:00

=== P2 TESTING (12/08–01/09/2015) ===
Jumlah event: 0
Distribusi event:
 Series([], Name: count, dtype: Int64)
Tanggal aktual: NaT s.d. NaT


In [7]:
# Validasi ulang filter berdasarkan timestamp Unix milidetik (UTC).
# Batas kanan eksklusif agar tanggal akhir tetap dihitung seluruhnya.
p1_awal = int(pd.Timestamp("2015-05-23", tz="UTC").timestamp() * 1000)
p1_akhir = int(pd.Timestamp("2015-08-12", tz="UTC").timestamp() * 1000)
p2_awal = int(pd.Timestamp("2015-08-12", tz="UTC").timestamp() * 1000)
p2_akhir = int(pd.Timestamp("2015-09-02", tz="UTC").timestamp() * 1000)

p1_timestamp = events.loc[(events["timestamp"] >= p1_awal) &
                          (events["timestamp"] < p1_akhir)]
p2_timestamp = events.loc[(events["timestamp"] >= p2_awal) &
                          (events["timestamp"] < p2_akhir)]

assert len(p1_timestamp) == len(p1), "Jumlah P1 tidak konsisten"
assert len(p2_timestamp) == len(p2), "Jumlah P2 tidak konsisten"
print("Validasi timestamp berhasil.")
print("P1:", len(p1), "event; batas Unix:", p1_awal, "<= timestamp <", p1_akhir)
print("P2:", len(p2), "event; batas Unix:", p2_awal, "<= timestamp <", p2_akhir)


Validasi timestamp berhasil.
P1: 552018 event; batas Unix: 1432339200000 <= timestamp < 1439337600000
P2: 0 event; batas Unix: 1439337600000 <= timestamp < 1441152000000


PEMERIKSAAN AKHIR SEBELUM IMPORT KE NEO47

In [8]:
def ringkasan_periode(df, nama):
    print(f"\n===== {nama} =====")
    print("Total event:", len(df))
    print("Jumlah user unik:", df["visitorid"].nunique())
    print("Jumlah item unik:", df["itemid"].nunique())
    print("Jumlah transaction ID unik:",
          df["transactionid"].nunique())
    print("\nJumlah user per jenis event:")
    print(
        df.groupby("event")["visitorid"]
        .nunique()
        .sort_values(ascending=False)
    )

ringkasan_periode(p1, "P1 - Training")
ringkasan_periode(p2, "P2 - Testing")


===== P1 - Training =====
Total event: 552018
Jumlah user unik: 284864
Jumlah item unik: 111184
Jumlah transaction ID unik: 3670

Jumlah user per jenis event:
event
view           284289
addtocart        7850
transaction      2475
Name: visitorid, dtype: int64

===== P2 - Testing =====
Total event: 0
Jumlah user unik: 0
Jumlah item unik: 0
Jumlah transaction ID unik: 0

Jumlah user per jenis event:
Series([], Name: visitorid, dtype: int64)


PERIKSA KESIAPAN DATA UNTUK GRAF

In [9]:
print("P1 - baris duplikat:", p1.duplicated().sum())
print("P1 - user kosong:", p1["visitorid"].isna().sum())
print("P1 - item kosong:", p1["itemid"].isna().sum())

print("\nContoh data P1:")
display(p1.head())

print("\nJumlah interaksi per user di P1:")
display(
    p1.groupby("visitorid")
    .size()
    .describe()
    .to_frame("jumlah_event_per_user")
)

P1 - baris duplikat: 71
P1 - user kosong: 0
P1 - item kosong: 0

Contoh data P1:


,timestamp,visitorid,event,itemid,transactionid,datetime,month
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117000+00:00,2015-06
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164000+00:00,2015-06
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827000+00:00,2015-06
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914000+00:00,2015-06
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106000+00:00,2015-06



Jumlah interaksi per user di P1:


,jumlah_event_per_user
count,284864.000000
mean,1.937830
std,8.034045
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,1435.000000


## Koneksi ke Neo4j Desktop milik Rima (lokal Windows)

- **Nama instance di Neo4j Desktop:** `UTS GSD` (ini nama tampilan instance, **bukan** nama database atau hostname).
- **Database:** `neo4j`; **user:** `neo4j`; **URI Bolt:** `bolt://localhost:7687`.
- **Driver Python:** `neo4j==6.4.1`.
- Password dimasukkan saat runtime melalui `getpass` atau environment variable `NEO4J_PASSWORD`, **tidak disimpan di file**, supaya aman untuk GitHub.
- Jalankan di **Jupyter lokal pada laptop Windows** dengan instance `UTS GSD` sudah **Running**. Google Colab tidak dapat menjangkau `localhost` laptop secara langsung.
- Node/relasi lama `PERFORMED` dari uji 5 baris mungkin masih ada. Import baru menggunakan `VIEWED`, `ADDED_TO_CART`, `BOUGHT` tanpa menghapus data lama.


### Tes koneksi ke instance `UTS GSD` — jalankan sebelum impor atau analisis Neo4j.


In [10]:
from neo4j import GraphDatabase
from getpass import getpass
import os

# Sesuai screenshot Neo4j Desktop Rima
NEO4J_INSTANCE = "UTS GSD"   # nama instance pada aplikasi Neo4j Desktop
NEO4J_DATABASE = "neo4j"    # nama database sesungguhnya
URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD") or getpass("Password Neo4j instance UTS GSD: ")

# Jika cell ini diulang, tutup driver sebelumnya terlebih dahulu.
if "driver" in globals():
    try:
        driver.close()
    except Exception:
        pass

driver = GraphDatabase.driver(
    URI, auth=(USERNAME, PASSWORD), connection_timeout=15
)
try:
    driver.verify_connectivity()
    with driver.session(database=NEO4J_DATABASE) as session:
        record = session.run(
            "RETURN currentDatabase() AS db, 'Berhasil terhubung!' AS status"
        ).single()
    print(record["status"])
    print("Instance Neo4j Desktop:", NEO4J_INSTANCE)
    print("Database:", record["db"])
    print("Username:", USERNAME)
    print("Bolt URI:", URI)
except Exception as exc:
    driver.close()
    print("Koneksi gagal. Pastikan instance UTS GSD Running di Neo4j Desktop,")
    print("password benar, dan notebook dijalankan secara lokal, bukan Colab.")
    raise


Password Neo4j instance UTS GSD: ··········
Koneksi gagal. Pastikan instance UTS GSD Running di Neo4j Desktop,
password benar, dan notebook dijalankan secara lokal, bukan Colab.


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

In [ ]:
# Verifikasi struktur graf yang sudah ada pada instance UTS GSD
with driver.session(database=NEO4J_DATABASE) as session:
    print("Status:", session.run("RETURN 'Koneksi Python berhasil' AS pesan").single()["pesan"])
    rows = session.run("MATCH ()-[r]->() RETURN type(r) AS jenis, count(r) AS jumlah ORDER BY jumlah DESC")
    for row in rows:
        print(f"{row['jenis']}: {row['jumlah']}")
print("Catatan: PERFORMED adalah relasi uji sebelumnya; import UTS menggunakan VIEWED/ADDED_TO_CART/BOUGHT.")


In [ ]:
# Driver sengaja tetap terbuka karena digunakan kembali untuk impor dan analisis.
# Tutup dengan driver.close() setelah seluruh notebook selesai dijalankan.
print("Driver siap digunakan pada sel berikutnya.")


PASTIKAN DATA P1 SIAP

In [ ]:
print("Ukuran P1:", p1.shape)
print("Kolom:", p1.columns.tolist())
print("\nDistribusi event:")
print(p1["event"].value_counts())

PERSIAPKAN DATA DI PYTHON

In [ ]:
# Pastikan hanya memakai data P1
import pandas as pd

kolom_diperlukan = [
    "timestamp",
    "visitorid",
    "event",
    "itemid",
    "transactionid"
]

assert all(col in p1.columns for col in kolom_diperlukan)

# Buat salinan khusus untuk proses import
import_df = p1[kolom_diperlukan].copy()

# Buat ID unik untuk setiap baris event
import_df["eventId"] = range(len(import_df))
for col in ["timestamp", "visitorid", "itemid", "eventId"]:
    import_df[col] = import_df[col].astype("int64")

# Ubah transactionid kosong menjadi None
import_df["transactionid"] = (
    import_df["transactionid"].where(
        import_df["transactionid"].notna(), None
    )
)

print("Jumlah baris yang akan diimpor:", len(import_df))
print("Jumlah event ID unik:", import_df["eventId"].nunique())
print("\nDistribusi event:")
print(import_df["event"].value_counts())

IMPORT P1 DARI PYTHON KE NEO4J

In [ ]:
driver.verify_connectivity()
print("Koneksi Neo4j berhasil.")

PERIKSA JUMLAH NODE

In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH (n)
        RETURN count(n) AS jumlah_node
    """)
    print("Jumlah node sebelum import:", result.single()["jumlah_node"])

KODE IMPORT

In [ ]:
# Pastikan import_df sudah dibuat pada tahap sebelumnya.
# Kolom eventId harus unik untuk setiap baris.

from time import time

BATCH_SIZE = 5000

IMPORT_QUERY = """
UNWIND $rows AS row

MERGE (v:Visitor {visitorId: row.visitorid})
MERGE (i:Item {itemId: row.itemid})

FOREACH (_ IN CASE
    WHEN row.event = 'view' THEN [1]
    ELSE []
END |
    MERGE (v)-[r:VIEWED {eventId: row.eventId}]->(i)
    SET r.timestamp = row.timestamp
)

FOREACH (_ IN CASE
    WHEN row.event = 'addtocart' THEN [1]
    ELSE []
END |
    MERGE (v)-[r:ADDED_TO_CART {eventId: row.eventId}]->(i)
    SET r.timestamp = row.timestamp
)

FOREACH (_ IN CASE
    WHEN row.event = 'transaction' THEN [1]
    ELSE []
END |
    MERGE (v)-[r:BOUGHT {eventId: row.eventId}]->(i)
    SET r.timestamp = row.timestamp,
        r.transactionId = row.transactionid)
"""

total_rows = len(import_df)
start_time = time()

with driver.session(database=NEO4J_DATABASE) as session:
    for start in range(0, total_rows, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total_rows)

        # Konversi hanya batch yang sedang diproses
        batch_df = import_df.iloc[start:end].copy()
        # Neo4j menerima Python int dan None, bukan numpy.int64 / NaN.
        batch_df["transactionid"] = batch_df["transactionid"].where(
            batch_df["transactionid"].notna(), None
        )
        batch = [
            {
                "timestamp": int(r.timestamp),
                "visitorid": int(r.visitorid),
                "itemid": int(r.itemid),
                "eventId": int(r.eventId),
                "event": str(r.event),
                "transactionid": None if pd.isna(r.transactionid) else str(int(r.transactionid)),
            }
            for r in batch_df.itertuples(index=False)
        ]

        result = session.run(IMPORT_QUERY, rows=batch)
        result.consume()

        if end % 100000 < BATCH_SIZE or end == total_rows:
            elapsed = time() - start_time
            print(
                f"Progress: {end:,}/{total_rows:,} event "
                f"({end / total_rows:.1%}) | "
                f"Waktu: {elapsed:.1f} detik"
            )

print("Import P1 selesai!")

BUAT MATRIKS SPARSE DARI P1
Nilai 1 jika pengguna pernah berinteraksi dengan item, dan 0 jika tidak. Jika pengguna melakukan beberapa jenis interaksi dengan item yang sama, item tersebut tetap bernilai 1

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# Gunakan data P1 yang sudah difilter
vsm_data = p1[["visitorid", "itemid"]].drop_duplicates()

# Buat indeks pengguna dan item
user_ids = pd.Index(vsm_data["visitorid"].unique())
item_ids = pd.Index(vsm_data["itemid"].unique())

user_index = user_ids.get_indexer(vsm_data["visitorid"])
item_index = item_ids.get_indexer(vsm_data["itemid"])

# Bentuk matriks User–Item sparse
user_item_matrix = csr_matrix(
    (
        np.ones(len(vsm_data), dtype=np.int8),
        (user_index, item_index)
    ),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.int8
)

print("Ukuran matriks User–Item:", user_item_matrix.shape)
print("Jumlah interaksi user-item unik:", user_item_matrix.nnz)
print("Jumlah pengguna:", len(user_ids))
print("Jumlah item:", len(item_ids))
print("Kepadatan matriks:",
      f"{user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6%}")

BENTUK VSM PENGGUNA DAN VSM ITEM

In [ ]:
# Baris = pengguna, kolom = item
user_vsm = user_item_matrix

# Baris = item, kolom = pengguna
item_vsm = user_item_matrix.T.tocsr()

print("User VSM:", user_vsm.shape)
print("Item VSM:", item_vsm.shape)

VISUALISASIKAN CONTOH MATRIKS

In [ ]:
import matplotlib.pyplot as plt

sample_size = min(50, user_vsm.shape[0])
sample_matrix = user_vsm[:sample_size, :50].toarray()

plt.figure(figsize=(12, 6))
plt.spy(sample_matrix, markersize=2)
plt.title("User–Item VSM (sampel 50 pengguna × 50 item)")
plt.xlabel("Item index")
plt.ylabel("User index")
plt.show()

Titik-titik yang terlihat menunjukkan interaksi. Area kosong menunjukkan pasangan pengguna–item yang tidak memiliki interaksi dalam sampel tersebut

BANDINGKAN PROYEKSI USER DAN ITEM

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Sampel untuk demonstrasi, bukan seluruh dataset
n_users_sample = min(500, user_vsm.shape[0])
n_items_sample = min(500, item_vsm.shape[0])

user_sample = user_vsm[:n_users_sample]
item_sample = item_vsm[:n_items_sample]

user_similarity_sample = cosine_similarity(user_sample)
item_similarity_sample = cosine_similarity(item_sample)

print("User similarity sample:",
      user_similarity_sample.shape)

print("Item similarity sample:",
      item_similarity_sample.shape)

PILIH SAMPEL SECARA ACAK

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(42)

n_user_sample = min(100, user_vsm.shape[0])
n_item_sample = min(100, item_vsm.shape[0])

user_idx = np.sort(
    rng.choice(user_vsm.shape[0], n_user_sample, replace=False)
)
item_idx = np.sort(
    rng.choice(item_vsm.shape[0], n_item_sample, replace=False)
)

user_sample = user_vsm[user_idx]
item_sample = item_vsm[item_idx]

user_sim = cosine_similarity(user_sample)
item_sim = cosine_similarity(item_sample)

print("User–User similarity:", user_sim.shape)
print("Item–Item similarity:", item_sim.shape)

BUAT GRAF PROYEKSI DENGAN EDGE SIMILIARITY MINIMAL 0.1 DAN SELF LOOP DIHILANGKAN

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_projection(similarity, title, threshold=0.1):
    G = nx.Graph()
    n = similarity.shape[0]
    G.add_nodes_from(range(n))

    for i in range(n):
        for j in range(i + 1, n):
            score = similarity[i, j]
            if score >= threshold:
                G.add_edge(i, j, weight=float(score))

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)

    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=30,
        width=0.7
    )

    plt.title(
        f"{title} | {G.number_of_nodes()} node, "
        f"{G.number_of_edges()} edge"
    )
    plt.axis("off")
    plt.show()

    print("Jumlah node:", G.number_of_nodes())
    print("Jumlah edge:", G.number_of_edges())

    return G

G_user = plot_projection(user_sim, "User–User Projection")
G_item = plot_projection(item_sim, "Item–Item Projection")

PERIKSA SIMILIARITY APAKAH MEMANG NOL

In [ ]:
print("USER SIMILARITY")
print("Nilai maksimum:", user_sim.max())
print("Nilai minimum:", user_sim.min())

upper_user = user_sim[np.triu_indices_from(user_sim, k=1)]
print("Pasangan dengan similarity > 0:", (upper_user > 0).sum())
print("Pasangan dengan similarity >= 0.1:", (upper_user >= 0.1).sum())

print("\nITEM SIMILARITY")
upper_item = item_sim[np.triu_indices_from(item_sim, k=1)]
print("Nilai maksimum:", item_sim.max())
print("Pasangan dengan similarity > 0:", (upper_item > 0).sum())
print("Pasangan dengan similarity >= 0.1:", (upper_item >= 0.1).sum())

COBA TRESHOLD LEBIH RENDAH

In [ ]:
G_user = plot_projection(
    user_sim,
    "User–User Projection",
    threshold=0.01
)

G_item = plot_projection(
    item_sim,
    "Item–Item Projection",
    threshold=0.01
)

PILIH USER DAN ITEM YANG PALING AKTIF

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Pilih 100 user paling aktif
top_users = (
    vsm_data.groupby("visitorid")["itemid"]
    .nunique()
    .nlargest(100)
    .index
)

user_idx = user_ids.get_indexer(top_users)
user_sample = user_vsm[user_idx]
user_sim = cosine_similarity(user_sample)

# 2. Pilih 100 item paling populer
top_items = (
    vsm_data.groupby("itemid")["visitorid"]
    .nunique()
    .nlargest(100)
    .index
)

item_idx = item_ids.get_indexer(top_items)
item_sample = item_vsm[item_idx]
item_sim = cosine_similarity(item_sample)

# 3. Periksa similarity antarobjek yang berbeda
upper_user = user_sim[np.triu_indices_from(user_sim, k=1)]
upper_item = item_sim[np.triu_indices_from(item_sim, k=1)]

print("USER SIMILARITY")
print("Jumlah pasangan similarity > 0:", (upper_user > 0).sum())
print("Jumlah pasangan similarity >= 0.1:", (upper_user >= 0.1).sum())
print("Similarity maksimum antar-user:", upper_user.max())

print("\nITEM SIMILARITY")
print("Jumlah pasangan similarity > 0:", (upper_item > 0).sum())
print("Jumlah pasangan similarity >= 0.1:", (upper_item >= 0.1).sum())
print("Similarity maksimum antar-item:", upper_item.max())

BUAT VISUALISASI PROYEKSI

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_projection(sim_matrix, title, threshold=0.1):
    G = nx.Graph()
    G.add_nodes_from(range(sim_matrix.shape[0]))

    for i in range(sim_matrix.shape[0]):
        for j in range(i + 1, sim_matrix.shape[1]):
            similarity = sim_matrix[i, j]
            if similarity >= threshold:
                G.add_edge(i, j, weight=float(similarity))

    plt.figure(figsize=(10, 7))
    pos = nx.spring_layout(G, seed=42)

    nx.draw_networkx_nodes(
        G, pos,
        node_size=80
    )
    nx.draw_networkx_edges(
        G, pos,
        alpha=0.5,
        width=1
    )

    plt.title(
        f"{title} (threshold={threshold})\n"
        f"Node: {G.number_of_nodes()}, "
        f"Edge: {G.number_of_edges()}"
    )
    plt.axis("off")
    plt.show()

    print(f"{title}")
    print("Jumlah node:", G.number_of_nodes())
    print("Jumlah edge:", G.number_of_edges())

    return G

G_user = plot_projection(
    user_sim, "Proyeksi User-User", threshold=0.1
)

G_item = plot_projection(
    item_sim, "Proyeksi Item-Item", threshold=0.1
)

BUAT DATA PASANGAN USER YANG MIRIP

In [ ]:
import numpy as np
import pandas as pd

THRESHOLD = 0.1

# Ambil hanya segitiga atas agar tidak menyimpan A-B dan B-A dua kali
i_idx, j_idx = np.triu_indices_from(user_sim, k=1)

mask = user_sim[i_idx, j_idx] >= THRESHOLD

user_similarity_df = pd.DataFrame({
    "user1": user_ids[user_idx[i_idx[mask]]],
    "user2": user_ids[user_idx[j_idx[mask]]],
    "similarity": user_sim[i_idx[mask], j_idx[mask]]
})

print("Jumlah relasi User–User:", len(user_similarity_df))
display(user_similarity_df.head(10))

SIMPAN KE NEO4J

In [ ]:
SAVE_USER_SIMILARITY = """
UNWIND $rows AS row
MATCH (u1:Visitor {visitorId: row.user1})
MATCH (u2:Visitor {visitorId: row.user2})
MERGE (u1)-[r:SIMILAR_TO]->(u2)
SET r.similarity = row.similarity
"""

records = user_similarity_df.to_dict("records")

with driver.session(database=NEO4J_DATABASE) as session:
    for start in range(0, len(records), 1000):
        session.run(
            SAVE_USER_SIMILARITY,
            rows=records[start:start + 1000]
        ).consume()

print("Relasi similarity User–User berhasil disimpan.")

MEMBUAT REKOMENDASI TOP 5 DENGAN KNN

SIAPKAN MATRIKS TETANGGA

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=6,
    n_jobs=-1
)

knn.fit(user_vsm)
print("Model KNN berhasil dibuat.")

PILIH PENGGUNA AKTIF

In [ ]:
user_activity = np.asarray(user_vsm.sum(axis=1)).ravel()
active_idx = np.flatnonzero(user_activity >= 5)

example_idx = active_idx[0]
example_user_id = user_ids[example_idx]

print("Contoh visitorId:", example_user_id)
print("Jumlah item yang pernah diinteraksikan:",
      user_activity[example_idx])

CARI TETANGGA TERDEKAT UNTUK SATU PENGGUNA

In [ ]:
distances, indices = knn.kneighbors(
    user_vsm[example_idx],
    n_neighbors=6
)

neighbors = []

for distance, idx in zip(distances[0], indices[0]):
    if idx == example_idx:
        continue

    neighbors.append({
        "visitorid": int(user_ids[idx]),
        "similarity": float(1 - distance)
    })

neighbors_df = pd.DataFrame(neighbors)

display(neighbors_df)

BUAT REKOMENDASI ITEM DARI TETANGGA

In [ ]:
from collections import defaultdict

scores = defaultdict(float)

for neighbor in neighbors:
    neighbor_idx = user_ids.get_loc(neighbor["visitorid"])
    interacted_items = user_vsm[neighbor_idx].indices

    for item_idx_value in interacted_items:
        scores[item_idx_value] += neighbor["similarity"]

seen_items = set(user_vsm[example_idx].indices)

recommendations = [
    (idx, score)
    for idx, score in scores.items()
    if idx not in seen_items
]

recommendations.sort(key=lambda x: x[1], reverse=True)

top5 = recommendations[:5]

recommendation_df = pd.DataFrame({
    "itemid": [int(item_ids[idx]) for idx, _ in top5],
    "score": [score for _, score in top5]
})

display(recommendation_df)

SIAPKAN MODEL KNN

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

# Hitung jumlah item unik yang diinteraksikan setiap user
user_activity = np.asarray(user_vsm.sum(axis=1)).ravel()

# Pilih user yang memiliki setidaknya 5 interaksi
active_idx = np.flatnonzero(user_activity >= 5)

print("Jumlah user dengan minimal 5 interaksi:", len(active_idx))

PILIH SATU PENGGUNA

In [ ]:
example_idx = active_idx[0]
example_user_id = int(user_ids[example_idx])

print("Visitor ID:", example_user_id)
print("Jumlah item yang pernah diinteraksikan:",
      int(user_activity[example_idx]))

CARI 5 TETANGGA TERDEKAT

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# user_idx berisi indeks 100 pengguna sampel teraktif
candidate_user_matrix = user_vsm[user_idx]

similarities = cosine_similarity(
    user_vsm[example_idx],
    candidate_user_matrix
).ravel()

# Urutkan dari similarity terbesar
order = np.argsort(similarities)[::-1]

neighbors = []

for pos in order:
    candidate_idx = user_idx[pos]

    # Jangan merekomendasikan pengguna itu sendiri
    if candidate_idx == example_idx:
        continue

    if similarities[pos] > 0:
        neighbors.append({
            "visitorid": int(user_ids[candidate_idx]),
            "similarity": float(similarities[pos]),
            "matrix_index": int(candidate_idx)
        })

    if len(neighbors) == 5:
        break

neighbors_df = pd.DataFrame(neighbors)
display(neighbors_df)

HASILKAN REKOMENDASI ITEM TOP 5

In [ ]:
from collections import defaultdict

scores = defaultdict(float)

for neighbor in neighbors:
    neighbor_idx = neighbor["matrix_index"]
    interacted_items = user_vsm[neighbor_idx].indices

    for item_idx_value in interacted_items:
        scores[item_idx_value] += neighbor["similarity"]

# Item yang sudah pernah dilihat atau diinteraksikan user target
seen_items = set(user_vsm[example_idx].indices)

recommendations = [
    (idx, score)
    for idx, score in scores.items()
    if idx not in seen_items
]

recommendations.sort(key=lambda x: x[1], reverse=True)

top5 = recommendations[:5]

recommendation_df = pd.DataFrame({
    "itemid": [int(item_ids[idx]) for idx, _ in top5],
    "score": [float(score) for _, score in top5]
})

display(recommendation_df)

EVALUASI MENGGUNAKAN P2

In [ ]:
print("Jumlah baris P2:", len(p2))
print("Kolom:", p2.columns.tolist())

PILIH TARGET USER DAN CARI 5 TETANGGA TERDEKAT

In [ ]:

import numpy as np
import pandas as pd

# Cari user sampel dengan total similarity terbesar
sim_no_diag = user_sim.copy()
np.fill_diagonal(sim_no_diag, 0)

target_pos = int(np.argmax(sim_no_diag.sum(axis=1)))
example_idx = int(user_idx[target_pos])
example_user_id = int(user_ids[example_idx])

print("Target Visitor ID:", example_user_id)
print("Jumlah item unik di data training:",
      int(user_vsm[example_idx].nnz))
print("Jumlah user serupa dalam sampel:",
      int(np.sum(user_sim[target_pos] > 0) - 0))


AMBIL 5 TETANGGA DENGAN SIMILIARITY TERBESAR (TIDAK TERMASUK DIRINYA SENDIRI)

In [ ]:

neighbor_order = np.argsort(user_sim[target_pos])[::-1]

neighbors = []
for pos in neighbor_order:
    candidate_idx = int(user_idx[pos])

    if candidate_idx == example_idx:
        continue

    sim = float(user_sim[target_pos, pos])
    if sim <= 0:
        continue

    neighbors.append({
        "visitorid": int(user_ids[candidate_idx]),
        "similarity": sim,
        "matrix_index": candidate_idx
    })

    if len(neighbors) == 5:
        break

neighbors_df = pd.DataFrame(neighbors)
display(neighbors_df)

print("Jumlah tetangga yang ditemukan:", len(neighbors_df))


BUAT REKOMENDASI 3 ITEM

In [ ]:

from collections import defaultdict

def recommend_for_sample_position(target_sample_pos, n_recommendations=5):
    target_matrix_idx = int(user_idx[target_sample_pos])
    similarity_row = user_sim[target_sample_pos]

    neighbor_positions = np.argsort(similarity_row)[::-1]
    scores = defaultdict(float)
    used_neighbors = 0

    for pos in neighbor_positions:
        neighbor_matrix_idx = int(user_idx[pos])

        if neighbor_matrix_idx == target_matrix_idx:
            continue

        similarity = float(similarity_row[pos])
        if similarity <= 0:
            continue

        # Tambahkan skor similarity untuk setiap item milik tetangga
        neighbor_items = user_vsm[neighbor_matrix_idx].indices
        for item_matrix_idx in neighbor_items:
            scores[int(item_matrix_idx)] += similarity

        used_neighbors += 1
        if used_neighbors == 5:
            break

    # Hilangkan item yang sudah pernah diinteraksikan target
    seen_items = set(user_vsm[target_matrix_idx].indices)
    ranked_items = [
        (item_idx, score)
        for item_idx, score in scores.items()
        if item_idx not in seen_items
    ]
    ranked_items.sort(key=lambda x: x[1], reverse=True)

    return [
        {
            "itemid": int(item_ids[item_idx]),
            "score": float(score)
        }
        for item_idx, score in ranked_items[:n_recommendations]
    ]

recommendation_df = pd.DataFrame(
    recommend_for_sample_position(target_pos, n_recommendations=3)
)

print("Rekomendasi untuk Visitor ID:", example_user_id)
display(recommendation_df)


SIMPAN SIMILIARITY ITEM ITEM KE NEO4J

In [ ]:

# Ambil pasangan unik dari matriks similarity item
item_rows, item_cols = np.where(
    np.triu(item_sim, k=1) >= 0.1
)

item_similarity_df = pd.DataFrame({
    "item1": [int(item_ids[item_idx[i]]) for i in item_rows],
    "item2": [int(item_ids[item_idx[j]]) for j in item_cols],
    "similarity": [float(item_sim[i, j])
                   for i, j in zip(item_rows, item_cols)]
})

print("Jumlah pasangan Item–Item:", len(item_similarity_df))
display(item_similarity_df.head(10))


SIMPAN KE NEO4J

In [ ]:

SAVE_ITEM_SIMILARITY = """
UNWIND $rows AS row
MATCH (i1:Item {itemId: row.item1})
MATCH (i2:Item {itemId: row.item2})
MERGE (i1)-[r:SIMILAR_TO]->(i2)
SET r.similarity = row.similarity
"""

item_rows_to_save = item_similarity_df.to_dict("records")

with driver.session(database=NEO4J_DATABASE) as session:
    for start in range(0, len(item_rows_to_save), 1000):
        batch = item_rows_to_save[start:start + 1000]
        session.run(SAVE_ITEM_SIMILARITY, rows=batch).consume()

print("Relasi Item–Item berhasil diproses.")


SIAPKAN DATA GROUND TRUTH P2
Agar pengujian mengukur kemampuan menemukan item baru, kita keluarkan item yang sudah pernah diinteraksikan oleh target pada P1 dari ground truth evaluasi.

In [ ]:

# Salin kolom yang diperlukan saja
p2_eval = p2[["visitorid", "itemid"]].drop_duplicates().copy()

# Pastikan ID konsisten sebagai integer
p2_eval["visitorid"] = p2_eval["visitorid"].astype("int64")
p2_eval["itemid"] = p2_eval["itemid"].astype("int64")

# Item yang dikenal selama training
known_items = set(map(int, item_ids))

# Hanya evaluasi item P2 yang tersedia di katalog P1
p2_eval = p2_eval[p2_eval["itemid"].isin(known_items)]

# Bentuk ground truth per user
ground_truth = (
    p2_eval.groupby("visitorid")["itemid"]
    .apply(set)
    .to_dict()
)

print("Jumlah user unik di P2 setelah filter item:", len(ground_truth))
print("Jumlah pasangan user-item P2 yang dievaluasi:", len(p2_eval))


Hitung Hit@5 dan NDCG@5

In [ ]:

def calculate_hit_ndcg(recommended_items, relevant_items, k=5):
    ranked = recommended_items[:k]

    # Hit@K = 1 jika ada minimal satu item relevan
    hit = int(any(item in relevant_items for item in ranked))

    # NDCG@K dengan relevansi biner
    dcg = sum(
        1.0 / np.log2(rank + 2)
        for rank, item in enumerate(ranked)
        if item in relevant_items
    )

    ideal_hits = min(len(relevant_items), k)
    idcg = sum(
        1.0 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0
    return hit, ndcg


evaluation_rows = []

# Gunakan user yang sama dengan sampel similarity
sample_user_ids = [int(user_ids[idx]) for idx in user_idx]

for sample_pos, visitor_id in enumerate(sample_user_ids):
    if visitor_id not in ground_truth:
        continue

    target_matrix_idx = int(user_idx[sample_pos])
    seen_items = set(map(int, user_vsm[target_matrix_idx].indices))
    # Konversi indeks kolom VSM ke item ID yang sebenarnya
    seen_item_ids = {int(item_ids[i]) for i in seen_items}

    relevant_items = ground_truth[visitor_id] - seen_item_ids

    # Ground truth kosong tidak memberi informasi untuk evaluasi ranking
    if not relevant_items:
        continue

    recs = recommend_for_sample_position(
        sample_pos,
        n_recommendations=5
    )
    recommended_items = [row["itemid"] for row in recs]

    hit5, ndcg5 = calculate_hit_ndcg(
        recommended_items,
        relevant_items,
        k=5
    )

    evaluation_rows.append({
        "visitorid": visitor_id,
        "n_relevant_test_items": len(relevant_items),
        "n_recommended": len(recommended_items),
        "Hit@5": hit5,
        "NDCG@5": ndcg5
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df.head(10))

if len(evaluation_df) > 0:
    print("Jumlah user yang berhasil dievaluasi:", len(evaluation_df))
    print("Rata-rata Hit@5:", evaluation_df["Hit@5"].mean())
    print("Rata-rata NDCG@5:", evaluation_df["NDCG@5"].mean())
else:
    print(
        "Belum ada user dengan ground truth P2 yang memenuhi kriteria. "
        "Periksa cakupan user dan item."
    )


SIMPAN HASIL

In [ ]:

# Simpan tabel hasil ke CSV
neighbors_df.to_csv("knn_neighbors.csv", index=False)
recommendation_df.to_csv("recommendations_top3.csv", index=False)
item_similarity_df.to_csv("item_similarity_sample.csv", index=False)
evaluation_df.to_csv("evaluation_p2_sample.csv", index=False)

print("File hasil analisis berhasil disimpan:")
print("- knn_neighbors.csv")
print("- recommendations_top3.csv")
print("- item_similarity_sample.csv")
print("- evaluation_p2_sample.csv")
